# Practical 08 – Support Vector Machines and Classifier Implementation

**Course:** ML6402 – Machine Learning  
**Instructor:** Dr. Ahmed Sabbah  
**University:** Al-Quds Open University


In [ ]:
from pathlib import Path
import pandas as pd

DATA_FILE = "Alexa_dataset.csv"
LOCAL_CANDIDATES = [Path("data") / DATA_FILE, Path("Lecture08_SVM_and_Classifier_Implementation") / "data" / DATA_FILE]
GITHUB_URL = "https://raw.githubusercontent.com/asabbah44/ML6402-Machine-Learning-Practicals/main/Lecture08_SVM_and_Classifier_Implementation/data/Alexa_dataset.csv"
local_path = next((p for p in LOCAL_CANDIDATES if p.exists()), None)
data_source = local_path if local_path else GITHUB_URL
df = pd.read_csv(data_source)
print(f"Loaded from: {data_source}")
df.head()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

X = df[["Age", "Minutes of music consumed"]]
y = df["Purchased Alexa"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

models = {"Logistic": LogisticRegression(),
          "SVM-linear": SVC(kernel="linear"),
          "SVM-RBF": SVC(kernel="rbf")}
for name, estimator in models.items():
    clf=Pipeline([("scale",StandardScaler()),("model",estimator)])
    clf.fit(X_train,y_train)
    pred=clf.predict(X_test)
    print("\n" + name)
    print("Accuracy :", round(accuracy_score(y_test, pred), 3))
    print("Precision:", round(precision_score(y_test, pred), 3))
    print("Recall   :", round(recall_score(y_test, pred), 3))
    print("F1       :", round(f1_score(y_test, pred), 3))
    print("Confusion matrix:\n", confusion_matrix(y_test, pred))

## Exercise
Try different `C` and `gamma` values for the RBF SVM and record the test F1 score.

## Source Chapter Visualization – Linear and RBF SVM

The author chapter includes training/test decision-region plots for both linear and nonlinear SVM. The following code reproduces that worked example with the course pipelines.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_svm_regions(model, X_plot, y_plot, title):
    x1_min, x1_max = X_plot.iloc[:, 0].min() - 2, X_plot.iloc[:, 0].max() + 2
    x2_min, x2_max = X_plot.iloc[:, 1].min() - 5000, X_plot.iloc[:, 1].max() + 5000

    xx1, xx2 = np.meshgrid(
        np.linspace(x1_min, x1_max, 250),
        np.linspace(x2_min, x2_max, 250)
    )

    grid = pd.DataFrame({
        X_plot.columns[0]: xx1.ravel(),
        X_plot.columns[1]: xx2.ravel()
    })

    zz = model.predict(grid).reshape(xx1.shape)

    plt.contourf(xx1, xx2, zz, alpha=0.25)
    plt.scatter(X_plot.iloc[:, 0], X_plot.iloc[:, 1], c=y_plot, edgecolor="k")
    plt.xlabel(X_plot.columns[0])
    plt.ylabel(X_plot.columns[1])
    plt.title(title)
    plt.show()

for name in ["SVM-linear", "SVM-RBF"]:
    estimator = models[name]
    svm_model = Pipeline([
        ("scale", StandardScaler()),
        ("model", estimator)
    ])
    svm_model.fit(X_train, y_train)

    plot_svm_regions(
        svm_model, X_train, y_train,
        f"{name} – Training Set"
    )
    plot_svm_regions(
        svm_model, X_test, y_test,
        f"{name} – Test Set"
    )

## Text-classification source example

The author classifier chapter also contains a separate IMDB text-classification example using:

- text cleaning;
- stop-word removal;
- stemming;
- CountVectorizer;
- TF-IDF;
- Naïve Bayes;
- Random Forest.

However, the referenced file `IMDB Dataset.csv` is **not included in the author's GitHub repository**. Because the source data is missing, this course repository does not invent or silently substitute another sentiment dataset.

If the original IMDB CSV is later added to the source repository, this example can be included exactly.